# TASK-031 — Qwen image–language interaction probe (inference only)

**No GPU execution is authorized by this coding task.** After separate owner authorization, choose a Colab L4, review Configuration and use Runtime → Run all. Default authorization is OFF. No training, optimizer, backward, generation, outer-fold scoring, Development or confirmation access.

Reuse `MyDrive/exp023_handoff.zip` and the existing `MyDrive/exp027/pilot_run_4bc703af144e/` checkpoint directory. No new input package is required. Only identity.json and the three selected models/seals are copied, read-only, into /content; original artifacts remain untouched. Do not point CHECKPOINT_ROOT to a different experiment.

Frozen manifest:32 inner-training paintings,48 fully observed rectangles,134 cells, three saved models, four prompt/head conditions and adapter-on/off same-head interventions.996 decoder sequences +36 repeat sequences +2 official parity sequences;198 image decoder batches +2 parity calls, up to34 cold vision forwards.3,216 raw logits. The adapter-off head and neutral/fixed/crossed conditions are OOD, not deployment estimates or a unique causal decomposition. Training-label alignment is descriptive, not generalization.

Planning only:1–4 sequences/s implies about4–17 minutes of decoder work, plus setup/vision/export. **No TASK-031 L4 timing has been measured.** Maximum30 allocation/wall minutes since Run all includes180s export reserve. Cold setup may exhaust the allowance; stop without altering settings. Complete image batches persist in Drive and resume only under identical source/model/input/runtime/GPU identity. The first batch per model/state is replayed exactly on resume. Corruption fails closed.

Outputs: raw keyed logits with SHA receipts, source/model/input identity, parity/determinism and timing/VRAM diagnostics, rectangle-level analysis and bootstrap summaries. Offline CPU verification checks integrity and recomputes analysis; it does not replay the encoder. Inspect H1/H2/H3 evidence; no mechanism conclusion is forced automatically. E50+L remains production.

Verified persistent export precedes the optional browser download and runtime release on success or controlled failure. An API return does not prove browser arrival or physical deallocation. Abrupt VM loss preserves already sealed batches; an interrupted unsealed publication is invalid and must not be treated as complete. See docs/task031_protocol.md for the frozen design.


In [ ]:
# OWNER_CONFIGURATION â€” fresh separate authorization is required for any GPU execution.
AUTHORIZE_EXPERIMENT = False
AUTHORIZE_GUARDED_CONFIRMATION = False
EXECUTION_SCOPE = "RESEARCH_ONLY_CV"  # Repository research-scope enum; this notebook runs pilot only.
RESOURCE_ELIGIBILITY_STATUS = "RESEARCH_ONLY_COMPETITION_UNCONFIRMED"
MAX_GPU_HOURS = 0.5
MAX_TOTAL_WALL_HOURS = 0.5
REQUIRE_SUBMISSION_ZIP = False
EVAL_INPUT_CSV = None
EVAL_DATA_DIR = None
AUTO_BROWSER_DOWNLOAD = True
AUTO_RELEASE_COLAB = True
INPUT_ZIP = "/content/drive/MyDrive/exp023_handoff.zip"
PERSISTENT_ROOT = "/content/drive/MyDrive/task031"
FEATURE_CACHE_ROOT = "/content/drive/MyDrive/exp026/features"
MODEL_CACHE_ROOT = "/content/exp026/model"
LOCAL_ROOT = "/content/task031"
STAGE = "probe"
SOURCE_REVISION = "8d2d24b6b83b31fdf5164da6717b13c0f94c9bf7"
ZIP_SHA256 = "2db4a53679940c7eb63c7c8272b91be8ec0618bb6c04a69ebaa15909e20f64e0"
MANIFEST_SHA256 = "b5315db3847c29c432e91510a3d8f99c4b1bf13aca8b17282a53059347a20aa9"
CHECKPOINT_ROOT = "/content/drive/MyDrive/exp027/pilot_run_4bc703af144e"


In [ ]:
import csv, hashlib, json, math, os, subprocess, sys, time, traceback, zipfile
from pathlib import Path


def stage_command(python, source, inputs, persistent, local, settings, remaining):
    if settings["STAGE"] != "probe":
        raise ValueError("Only the frozen inference probe is supported. Training/outer-fold/CV/Development stages are forbidden.")
    return [python, "-u", str(source/"scripts/run_task031.py"), "--stage", settings["STAGE"],
            "--inputs", str(inputs), "--checkpoints", str(local/"checkpoints"), "--cache", str(settings["FEATURE_CACHE_ROOT"]),
            "--run-dir", str(persistent/("probe_run_"+settings["SOURCE_REVISION"][:12])),
            "--model-cache", str(settings["MODEL_CACHE_ROOT"]), "--max-seconds", str(remaining), "--authorize-gpu"]


def probe_notebook(settings, drive, files, runtime):
    started = time.monotonic()
    local, persistent = Path(settings["LOCAL_ROOT"]), Path(settings["PERSISTENT_ROOT"])
    attempt = persistent/"attempts"/str(time.time_ns())
    result = {"status":"STARTING", "research_only":True, "production":"E50+L unchanged",
              "browser_arrival":"UNVERIFIED", "physical_GPU_deallocation":"UNVERIFIED"}
    mounted = False
    run_directory = persistent/("probe_run_"+settings["SOURCE_REVISION"][:12])
    source = local/"source"

    def remaining():
        value = float(min(settings["MAX_GPU_HOURS"], settings["MAX_TOTAL_WALL_HOURS"]))*3600 - (time.monotonic()-started) - 180
        if value <= 30: raise TimeoutError("Allocation exhausted; completed persistent boundaries remain available")
        return value

    def run(command, name, env=None, timeout=None):
        print("Running", name, flush=True)
        log = attempt/(name+".log")
        with log.open("w") as handle:
            process = subprocess.run(command, stdout=handle, stderr=subprocess.STDOUT,
                                     timeout=remaining() if timeout is None else timeout, env=env)
        print("\n".join(log.read_text(errors="replace").splitlines()[-20:]))
        if process.returncode:
            raise RuntimeError(f"{name} stopped (exit {process.returncode}). Inspect {log}; do not change scientific settings to force PASS.")

    def checksum(path):
        with path.open("rb") as handle: return hashlib.file_digest(handle,"sha256").hexdigest()

    try:
        drive.mount("/content/drive"); mounted = True
        attempt.mkdir(parents=True,exist_ok=False); local.mkdir(parents=True,exist_ok=True)
        if not settings["AUTHORIZE_EXPERIMENT"]:
            raise PermissionError("No GPU work launched. Explicitly authorize this TASK-031 inference allocation in Configuration before Run all.")
        if settings["EXECUTION_SCOPE"]!="RESEARCH_ONLY_CV":
            raise ValueError("TASK-031 is research-only; competition eligibility remains unresolved.")
        if settings["AUTHORIZE_GUARDED_CONFIRMATION"] or settings["REQUIRE_SUBMISSION_ZIP"] or settings["EVAL_INPUT_CSV"] is not None or settings["EVAL_DATA_DIR"] is not None:
            raise ValueError("Confirmation, evaluation inputs and submission must remain disabled")
        if settings["STAGE"] != "probe":
            raise ValueError("Only the frozen inference probe is supported. Training/outer-fold/CV/Development stages are forbidden.")
        for limit in (settings["MAX_GPU_HOURS"],settings["MAX_TOTAL_WALL_HOURS"]):
            if not math.isfinite(float(limit)) or not 0<float(limit)<=0.5:
                raise ValueError("Allocation limits must be positive, finite and at most0.5 hours")
        if settings["RESOURCE_ELIGIBILITY_STATUS"]!="RESEARCH_ONLY_COMPETITION_UNCONFIRMED":
            raise ValueError("Only the declared research-only resource scope is supported; no competition eligibility is inferred")
        if checksum(Path(settings["INPUT_ZIP"]))!=settings["ZIP_SHA256"]:
            raise ValueError("Input ZIP SHA-256 differs. Use the unchanged TASK-025C exp023_handoff.zip.")
        if not source.exists():
            run(["git","clone","--no-checkout","https://github.com/bqa100507-spec/SemEval-2027-Task-12---ArtEmotion-28-Subtask-3---Image-Only-Cross-Cultural-Sentiment-Prediction.git",str(source)],"clone")
            run(["git","-C",str(source),"checkout","--detach",settings["SOURCE_REVISION"]],"checkout")
        if subprocess.check_output(["git","-C",str(source),"rev-parse","HEAD"],text=True).strip()!=settings["SOURCE_REVISION"]:
            raise ValueError("Existing source differs. Choose a new LOCAL_ROOT; do not overwrite historical runs.")
        if subprocess.check_output(["git","-C",str(source),"status","--porcelain","--untracked-files=no"],text=True).strip():
            raise ValueError("Pinned tracked source was modified; restore a clean checkout in a fresh LOCAL_ROOT.")
        evidence = json.loads((source/"docs/exp027_closure_evidence.json").read_text())["files_sha256"]
        checkpoint_names = ["identity.json", *["models/"+m+suffix for m in ("BCE","P_low","P_high") for suffix in (".pt",".sha256.json")]]
        for name in checkpoint_names:
            expected_checkpoint = evidence.get(name, evidence.get(name.replace("/",chr(92))))
            original = Path(settings["CHECKPOINT_ROOT"])/name
            if expected_checkpoint is None or checksum(original)!=expected_checkpoint:
                raise ValueError("Missing/changed EXP-027 selected checkpoint: "+str(original))
            destination = local/"checkpoints"/name
            destination.parent.mkdir(parents=True,exist_ok=True)
            if destination.exists() and checksum(destination)!=expected_checkpoint:
                raise ValueError("Existing local checkpoint changed; choose a fresh LOCAL_ROOT")
            if not destination.exists():
                temporary = destination.with_suffix(destination.suffix+".pending")
                temporary.write_bytes(original.read_bytes());temporary.replace(destination)
            if checksum(destination)!=expected_checkpoint:raise IOError("Checkpoint copy read-back differs")
        config = json.loads((source/"configs/exp026.json").read_text())
        expected_inputs = {k:v for k,v in config["input_sha256"].items() if k in ("training/pairs.csv","training/protocol.csv")}
        inputs = local/"inputs"
        with zipfile.ZipFile(settings["INPUT_ZIP"]) as archive:
            manifest_raw = archive.read("input_manifest.json")
            if hashlib.sha256(manifest_raw).hexdigest()!=settings["MANIFEST_SHA256"]:raise ValueError("Pinned input manifest SHA-256 differs")
            protocol = archive.read("training/protocol.csv")
            if hashlib.sha256(protocol).hexdigest()!=config["input_sha256"]["training/protocol.csv"]:
                raise ValueError("Protocol checksum differs")
            rows = [row for row in csv.DictReader(protocol.decode().splitlines()) if row["role"]=="development"]
            if len(rows)!=1335: raise ValueError("Expected exactly1335 P2 original training paintings")
            targets = expected_inputs  # Only the two verified vote/protocol files; no captions, reference predictions or sealed annotations.
            # Extract only the two pinned training vote/protocol files here; later extract only the frozen probe images.
            for name, expected in targets.items():
                target = (inputs/name).resolve()
                if not target.is_relative_to(inputs.resolve()): raise ValueError("Unsafe ZIP path")
                if target.exists():
                    if checksum(target)!=expected: raise ValueError("Existing local input changed: "+name)
                    continue
                raw = archive.read(name)
                if hashlib.sha256(raw).hexdigest()!=expected: raise ValueError("Input checksum differs: "+name)
                target.parent.mkdir(parents=True,exist_ok=True)
                temporary = target.with_suffix(target.suffix+".pending")
                temporary.write_bytes(raw); temporary.replace(target)
        receipt = {"source_revision":settings["SOURCE_REVISION"], "input_zip_sha256":settings["ZIP_SHA256"], "input_manifest_sha256":settings["MANIFEST_SHA256"],
                   "input_sha256":expected_inputs, "stage":settings["STAGE"],
                   "source_sha256":{name:checksum(source/name) for name in ("configs/task031.json","configs/task031_manifest.json","docs/task031_protocol.md","scripts/run_task031.py","scripts/verify_task031.py","src/st3/interaction_probe.py","src/st3/probe_inference.py","requirements-exp026.lock.txt")}}
        (attempt/"identity.json").write_text(json.dumps(receipt,indent=2))
        env = dict(os.environ, UV_PYTHON_INSTALL_DIR=str(local/"python"), UV_CACHE_DIR=str(local/"uv_cache"), HF_HUB_DISABLE_XET="1")
        run([sys.executable,"-m","pip","install","uv==0.7.13"],"uv",env)
        run([sys.executable,"-m","uv","python","install","3.12.10"],"python",env)
        if not (local/"venv").exists():
            run([sys.executable,"-m","uv","venv","--python","3.12.10",str(local/"venv")],"venv",env)
        python = str(local/"venv/bin/python")
        run([sys.executable,"-m","uv","pip","sync","--python",python,"--extra-index-url","https://download.pytorch.org/whl/cu126",
             "--index-strategy","unsafe-best-match",str(source/"requirements-exp026.lock.txt")],"dependencies",env)
        run([sys.executable,"-m","uv","pip","check","--python",python],"dependency_check",env)
        run([python,"-c","import torch; assert torch.cuda.is_available(), 'Select a CUDA GPU'; print(torch.cuda.get_device_name()); assert 'L4' in torch.cuda.get_device_name(), 'Select a Colab L4'; assert torch.cuda.mem_get_info()[0]>10*1024**3, 'Need at least10GiB free VRAM'"],"GPU_check",env)
        # Worker has its own shorter hard deadline, so its child exits before this supervisor.
        # CPU/workflow checks run in the pinned interpreter with CUDA hidden; no model or outer targets.
        test_env = dict(env, CUDA_VISIBLE_DEVICES="-1")
        run([python,"-m","pytest","-q",str(source/"tests/test_interaction_probe.py"),str(source/"tests/test_experiment_workflow.py")],"CPU_workflow_checks",test_env)
        run([python,"-u",str(source/"scripts/run_task031.py"),"--stage","prepare","--inputs",str(inputs),"--checkpoints",str(local/"checkpoints"),"--run-dir",str(run_directory)],"probe_input_isolation",test_env)
        selected = json.loads((run_directory/"probe_manifest.json").read_text())
        print("Frozen forward budget:",json.dumps(selected["counts"],indent=2))
        with zipfile.ZipFile(settings["INPUT_ZIP"]) as archive:
            for row in selected["images"].values():
                name="artwork/images/"+row["image"];target=(inputs/name).resolve()
                if not target.is_relative_to(inputs.resolve()):raise ValueError("Unsafe artwork path")
                if target.exists():
                    if checksum(target)!=row["image_sha256"]:raise ValueError("Existing artwork checksum differs")
                    continue
                raw=archive.read(name)
                if hashlib.sha256(raw).hexdigest()!=row["image_sha256"]:raise ValueError("Artwork checksum differs")
                target.parent.mkdir(parents=True,exist_ok=True);temp=target.with_suffix(".pending");temp.write_bytes(raw);temp.replace(target)
        worker_seconds = remaining()-30
        run(stage_command(python,source,inputs,persistent,local,settings,worker_seconds),"experiment",env,timeout=worker_seconds+20)
        report = json.loads((run_directory/(settings["STAGE"]+"_report.json")).read_text())
        verified = json.loads((run_directory/"verification.json").read_text())
        if report["status"]!="INFERENCE_COMPLETE_OFFLINE_VERIFIED" or verified["status"]!="PASS":
            raise ValueError("Incomplete/failed probe cannot produce a mechanism result")
        result.update(report)
        print("Measured report:",json.dumps(report,indent=2))
        print("Inference-only probe:", report["status"], "-- descriptive training mechanism only; no promotion result.")
    except Exception as error:
        result.update(status="EXECUTION_STOP",error=f"{type(error).__name__}: {error}",traceback=traceback.format_exc())
    finally:
        try:
            if mounted:
                attempt.mkdir(parents=True,exist_ok=True)
                (attempt/"status.json").write_text(json.dumps(result,indent=2))
                output = attempt/"outputs.zip"
                manifest = {}
                with zipfile.ZipFile(output,"w",zipfile.ZIP_DEFLATED) as archive:
                    for directory,prefix in ((attempt,"attempt"),(run_directory,"run")):
                        if not directory.exists():continue
                        for path in sorted(directory.rglob("*")):
                            if not path.is_file() or path==output or ".pending" in path.name or ".partial" in path.name:continue
                            name = prefix+"/"+path.relative_to(directory).as_posix()
                            manifest[name]=checksum(path);archive.write(path,name)
                    archive.writestr("manifest_sha256.json",json.dumps(manifest,indent=2))
                with zipfile.ZipFile(output) as archive:
                    for name, expected in manifest.items():
                        if hashlib.sha256(archive.read(name)).hexdigest()!=expected:
                            raise IOError("Persistent output read-back failed: "+name)
                (attempt/"outputs.sha256").write_text(checksum(output)+"  outputs.zip\n")
                result["persistent_readback"]="PASS"
                result["feature_cache"]="Completed image caches persist separately under "+str(settings["FEATURE_CACHE_ROOT"])
                if settings["AUTO_BROWSER_DOWNLOAD"]:
                    result["download_request"]="REQUESTED"
                    result["download_sha256"]=checksum(output)
                    (attempt/"download_attempt.json").write_text(json.dumps({"status":"REQUESTED","sha256":result["download_sha256"],"arrival":"UNVERIFIED"}))
                    try:files.download(str(output));result["download_request"]="API_RETURNED"
                    except Exception as error:result["download_request"]="FAILED: "+str(error)
        except Exception as error:result.update(status_before_export=result.get("status"),status="FAILED_EXPORT",export_error=str(error),persistent_readback="FAIL")
        finally:
            result["runtime_release"]="REQUESTED_NOT_CONFIRMED" if settings["AUTO_RELEASE_COLAB"] else "DISABLED"
            if mounted:
                try:(attempt/"final_report.json").write_text(json.dumps(result,indent=2))
                except Exception:pass
            print(json.dumps(result,indent=2))
            if settings["AUTO_RELEASE_COLAB"]:
                try:runtime.unassign()
                except Exception as error:
                    result["runtime_release"]="FAILED: "+str(error)
                    if mounted:
                        try:(attempt/"final_report.json").write_text(json.dumps(result,indent=2))
                        except Exception:pass
                    print("Runtime release request failed:",error)
    return result


In [ ]:
from google.colab import drive, files, runtime
settings = {key: globals()[key] for key in ('AUTHORIZE_EXPERIMENT', 'AUTHORIZE_GUARDED_CONFIRMATION', 'EXECUTION_SCOPE', 'RESOURCE_ELIGIBILITY_STATUS', 'MAX_GPU_HOURS', 'MAX_TOTAL_WALL_HOURS', 'REQUIRE_SUBMISSION_ZIP', 'EVAL_INPUT_CSV', 'EVAL_DATA_DIR', 'AUTO_BROWSER_DOWNLOAD', 'AUTO_RELEASE_COLAB', 'INPUT_ZIP', 'PERSISTENT_ROOT', 'FEATURE_CACHE_ROOT', 'MODEL_CACHE_ROOT', 'LOCAL_ROOT', 'STAGE', 'SOURCE_REVISION', 'ZIP_SHA256', 'MANIFEST_SHA256', 'CHECKPOINT_ROOT')}
run_cloud = probe_notebook
result = run_cloud(settings, drive, files, runtime)
